# 🏆 Notebook 02 — Bradley-Terry Reward Model Training

**RLHF Preference Trainer** · Step 2 of 5

This notebook trains a **Bradley-Terry reward model** on the preference CSV from Notebook 01.

Architecture:
- **Backbone**: GPT-2 Medium (345M params)
- **Head**: Linear(1024 → 1) scalar reward
- **Loss**: `−E[log σ(r_chosen − r_rejected)]`
- **Optimizer**: AdamW, lr=2e-5, cosine schedule

The trained model is saved to `reward_model/` for use in Notebook 03.

> **Runtime**: T4 GPU recommended (~30–45 min for full dataset). CPU fallback available.

---

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────
!pip install -q transformers torch datasets scikit-learn pandas matplotlib accelerate
print("✅ Dependencies installed")

In [ ]:
# ── Cell 2: Path setup ───────────────────────────────────────────────────
import os, sys

if 'google.colab' in str(get_ipython()):
    if not os.path.exists('rlhf-preference-trainer'):
        !git clone https://github.com/sharma614/rlhf-preference-trainer.git
    os.chdir('rlhf-preference-trainer')

    # Mount Drive if preferences CSV is there
    from google.colab import drive
    try:
        drive.mount('/content/drive', force_remount=False)
        DRIVE_CSV = '/content/drive/MyDrive/rlhf_data/preferences.csv'
        if os.path.exists(DRIVE_CSV):
            os.makedirs('data', exist_ok=True)
            !cp "{DRIVE_CSV}" data/preferences.csv
            print(f"✅ Copied preferences.csv from Drive")
    except Exception as e:
        print(f"Drive not mounted or CSV not found: {e}")
else:
    project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
    os.chdir(project_root)

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print(f"📁 CWD: {os.getcwd()}")

In [ ]:
# ── Cell 3: Imports ──────────────────────────────────────────────────────
import torch
import pandas as pd
import matplotlib.pyplot as plt
from transformers import AutoTokenizer

from src.data_utils import load_preferences, build_reward_dataset, split_dataset
from src.reward_model import BradleyTerryRewardModel, train_reward_model, evaluate_reward_model, PreferenceDataset
from src.ppo_config import MODEL_NAME, PREFERENCES_CSV, REWARD_MODEL_DIR, SEED

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"🖥️  Device: {device}")
if device == 'cpu':
    print("⚠️  No GPU detected — training will be slow. Consider Colab T4 runtime.")

In [ ]:
# ── Cell 4: Generate synthetic preferences (if real CSV is missing) ───────
# This allows running the full pipeline even before annotation is complete.
# Replace with real data from Notebook 01 for best results.

import os
from src.data_utils import SEED_PROMPTS, init_preferences_csv, save_annotation
import datetime, random

SYNTHETIC_N = 200  # Increase to 1200 if you want full synthetic training

if not os.path.exists(PREFERENCES_CSV):
    print(f"⚠️  No real CSV found at {PREFERENCES_CSV}.")
    print(f"   Generating {SYNTHETIC_N} synthetic preference pairs for demo...")

    os.makedirs('data', exist_ok=True)
    init_preferences_csv(PREFERENCES_CSV)
    rng = random.Random(SEED)

    GOOD_RESPONSES = [
        "This is a comprehensive and accurate response that addresses the question clearly and provides useful context.",
        "Excellent explanation with clear examples, factual accuracy, and appropriate depth for the audience.",
        "Well-structured answer that covers the key points accurately without unnecessary filler or misinformation.",
        "The response is informative, safe, and fluently written with a logical flow of ideas.",
        "Helpful and accurate with good coverage of the topic, written clearly and coherently.",
    ]
    BAD_RESPONSES = [
        "I don't know. This question is too hard and I cannot answer it properly at all.",
        "The answer is complicated. There are many factors. It depends on various things.",
        "Response A response response response unclear unclear repetitive unclear unclear.",
        "This is wrong. Incorrect information that contradicts established scientific understanding.",
        "blah blah blah random words that do not form a coherent or helpful response.",
    ]

    for i in range(SYNTHETIC_N):
        prompt = rng.choice(SEED_PROMPTS)
        if rng.random() < 0.55:  # A is preferred 55% of time
            resp_a = rng.choice(GOOD_RESPONSES)
            resp_b = rng.choice(BAD_RESPONSES)
            preferred = 'A'
        else:
            resp_a = rng.choice(BAD_RESPONSES)
            resp_b = rng.choice(GOOD_RESPONSES)
            preferred = 'B'

        save_annotation({
            'prompt': prompt,
            'response_a': resp_a,
            'response_b': resp_b,
            'preferred': preferred,
            'helpfulness_score': rng.randint(3, 5) if preferred == 'A' else rng.randint(1, 3),
            'factuality_score': rng.randint(3, 5),
            'safety_score': 5,
            'fluency_score': rng.randint(3, 5),
            'annotator_id': f'synthetic_{i % 3}',
            'timestamp': datetime.datetime.now().isoformat(),
        }, PREFERENCES_CSV)

    print(f"✅ Generated {SYNTHETIC_N} synthetic pairs at {PREFERENCES_CSV}")
else:
    n = len(pd.read_csv(PREFERENCES_CSV))
    print(f"✅ Found real preferences CSV with {n} annotations")

In [ ]:
# ── Cell 5: Load and prepare dataset ────────────────────────────────────
df = load_preferences(PREFERENCES_CSV)
print(f"\nPreference distribution:")
print(df['preferred'].value_counts())

reward_data = build_reward_dataset(df)
train_data, eval_data = split_dataset(reward_data, test_size=0.1, seed=SEED)

print(f"\n✅ Dataset prepared:")
print(f"   Train: {len(train_data['chosen'])} pairs")
print(f"   Eval:  {len(eval_data['chosen'])} pairs")
print(f"\nExample chosen text (truncated):")
print(train_data['chosen'][0][:200] + '...')

In [ ]:
# ── Cell 6: Load tokenizer ────────────────────────────────────────────────
print(f"⏳ Loading tokenizer for {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
print(f"✅ Tokenizer loaded — vocab size: {tokenizer.vocab_size:,}")

In [ ]:
# ── Cell 7: Instantiate reward model ────────────────────────────────────
print(f"⏳ Instantiating BradleyTerryRewardModel ({MODEL_NAME})...")
reward_model = BradleyTerryRewardModel(
    model_name=MODEL_NAME,
    freeze_backbone=False,  # Fine-tune the whole model
)

total_params = sum(p.numel() for p in reward_model.parameters())
trainable_params = sum(p.numel() for p in reward_model.parameters() if p.requires_grad)
print(f"✅ Model instantiated")
print(f"   Total parameters    : {total_params:,}")
print(f"   Trainable parameters: {trainable_params:,}")
print(f"   Reward head size    : {reward_model.reward_head.weight.numel():,}")

In [ ]:
# ── Cell 8: Training configuration ──────────────────────────────────────
TRAIN_CONFIG = {
    'num_epochs': 3,
    'batch_size': 4 if device == 'cuda' else 2,
    'learning_rate': 2e-5,
    'max_length': 256,   # Reduced from 512 for Colab memory
    'grad_clip': 1.0,
    'log_every': max(1, len(train_data['chosen']) // (4 * 10)),  # ~10 logs per epoch
}

print("Training configuration:")
for k, v in TRAIN_CONFIG.items():
    print(f"  {k}: {v}")

estimated_steps = (len(train_data['chosen']) // TRAIN_CONFIG['batch_size']) * TRAIN_CONFIG['num_epochs']
print(f"\n  Estimated total steps: {estimated_steps}")

In [ ]:
# ── Cell 9: Train reward model ───────────────────────────────────────────
print("🚀 Starting reward model training...")
print(f"   This will take ~{TRAIN_CONFIG['num_epochs'] * 5}–{TRAIN_CONFIG['num_epochs'] * 15} minutes on T4.")
print()

logs = train_reward_model(
    model=reward_model,
    tokenizer=tokenizer,
    train_data=train_data,
    eval_data=eval_data,
    save_dir=REWARD_MODEL_DIR,
    **TRAIN_CONFIG,
)

print(f"\n✅ Training complete! Model saved to '{REWARD_MODEL_DIR}/'")

In [ ]:
# ── Cell 10: Plot training curves ────────────────────────────────────────
log_df = pd.DataFrame(logs)

if len(log_df) > 0:
    from src.evaluation import plot_reward_model_curves
    os.makedirs('evaluation_results', exist_ok=True)
    plot_reward_model_curves(
        log_df,
        save_path='evaluation_results/reward_model_curves.png',
        show=True
    )
else:
    print("⚠️  No logs to plot (log_every may be larger than total steps).")

In [ ]:
# ── Cell 11: Final evaluation ────────────────────────────────────────────
from src.reward_model import PreferenceDataset, evaluate_reward_model
import torch

eval_ds = PreferenceDataset(eval_data, tokenizer, max_length=256)
eval_loader = torch.utils.data.DataLoader(eval_ds, batch_size=4, shuffle=False)

final_acc = evaluate_reward_model(reward_model.to(device), eval_loader, device)
print(f"\n{'='*50}")
print(f"  REWARD MODEL FINAL RESULTS")
print(f"{'='*50}")
print(f"  Eval Accuracy (r_chosen > r_rejected): {final_acc:.4f}")
print(f"  Target accuracy: ~0.70+ (well above 0.50 random)")
if final_acc >= 0.65:
    print(f"  ✅ Model is learning preference signal!")
else:
    print(f"  ⚠️  Accuracy is low — consider more training data or more epochs.")
print(f"{'='*50}")

In [ ]:
# ── Cell 12: Score sample responses ─────────────────────────────────────
from src.reward_model import score_response

test_prompt = "Explain how neural networks learn from data."
good_response = "Neural networks learn through a process called backpropagation. During training, the network receives input data, makes a prediction, and computes how wrong that prediction is using a loss function. The gradient of this loss is then computed with respect to each parameter, and the parameters are updated in the direction that reduces the loss. This process repeats over many examples until the network learns useful patterns."
bad_response  = "Neural networks are complicated. They use math. The learning happens automatically somehow."

r_good = score_response(reward_model, tokenizer, test_prompt, good_response, device=device)
r_bad  = score_response(reward_model, tokenizer, test_prompt, bad_response,  device=device)

print(f"\nReward Scoring Demo:")
print(f"  Good response reward: {r_good:.4f}")
print(f"  Bad response reward : {r_bad:.4f}")
if r_good > r_bad:
    print(f"  ✅ Model correctly ranks the good response higher!")
else:
    print(f"  ⚠️  Model needs more training data to distinguish quality.")

In [ ]:
# ── Cell 13: Smoke test & next steps ────────────────────────────────────
import os
assert os.path.exists(REWARD_MODEL_DIR), f"ERROR: {REWARD_MODEL_DIR} not found!"
assert os.path.exists(f"{REWARD_MODEL_DIR}/reward_head.pt"), "ERROR: reward_head.pt missing!"
assert os.path.exists(f"{REWARD_MODEL_DIR}/training_log.csv"), "ERROR: training_log.csv missing!"

print("✅ Smoke test PASSED")
print(f"   Files in {REWARD_MODEL_DIR}/:")
for f in os.listdir(REWARD_MODEL_DIR):
    size = os.path.getsize(f"{REWARD_MODEL_DIR}/{f}") / 1024 / 1024
    print(f"     {f:40s} {size:.1f} MB")
print(f"\n   ✨ Next step: Run notebook 03_ppo_finetuning.ipynb")